# Lesson 2.3 — When Perception Lies

Three realistic faults on the real perception model: **occlusion**, **noise**, **duplicates**. Each is absorbed at the seam, not propagated into a bad pick. (Greenhouse arm: L1=0.4, L2=0.3, reach ≈ 0.7 m.)

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand)
checks = []
world = GreenhouseWorld(fruit=[
    Fruit('F1', 0.30, 0.40, ripe=True),
    Fruit('F2', 0.45, 0.00, ripe=True),
    Fruit('F4', 0.55, 0.10, ripe=True)],
    q=np.array([0.6, 0.8]))
tool = np.array([0.0, 0.0])


### Fault 1 — occlusion: the committed target disappears, system falls back

In [2]:
clean = model_perception(world, rng=np.random.default_rng(1))
_, base = understand(clean, world, tool_xy=tool)
print('baseline committed:', base['id'])
occ = model_perception(world, rng=np.random.default_rng(1), occlude=[base['id']])
_, fb = understand(occ, world, tool_xy=tool)
print('after occluding', base['id'], '-> committed:', fb['id'] if fb else None)
checks.append(fb is None or fb['id'] != base['id'])   # fell back (or nothing left)
checks.append(all(d['id'] != base['id'] for d in occ))  # occluded id truly absent


baseline committed: F2
after occluding F2 -> committed: F1


### Fault 2 — duplicates: world-state count does NOT grow

In [3]:
dup = model_perception(world, rng=np.random.default_rng(1), duplicate=['F2'])
t_clean, _ = understand(clean, world, tool_xy=tool)
t_dup, _ = understand(dup, world, tool_xy=tool)
print('raw with dup:', len(dup), '| world-state targets clean:', len(t_clean),
      'with dup:', len(t_dup))
checks.append(len(dup) == len(clean)+1)        # raw grew by the duplicate
checks.append(len(t_dup) == len(t_clean))      # world state did NOT grow


raw with dup: 4 | world-state targets clean: 3 with dup: 3


### Fault 3 — noise: committed target stays ripe & reachable

In [4]:
noisy = model_perception(world, rng=np.random.default_rng(2), noise=0.02)
_, nt = understand(noisy, world, tool_xy=tool)
print('under noise -> committed:', nt['id'], 'ripe=%s reachable=%s' % (nt['ripe'], nt['reachable']))
checks.append(nt is not None and nt['ripe'] and nt['reachable'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


under noise -> committed: F2 ripe=True reachable=True
5/5 checks passed.
All checks passed.
